In [1]:
import random

def play_game():
    score = 0
    for i in range(7):
        if random.random() < 0.5:  # heads
            score += 2
        else:  # tails
            score *= 2
    return score

num_trials = 1000000
total_score = sum(play_game() for _ in range(num_trials))
expected_score = total_score / num_trials

print(f"Expected score after 7 flips: {expected_score:.6f}")

Expected score after 7 flips: 32.142974


In [2]:
import random
import math

def play_game():
    score = 0
    for i in range(7):
        if random.random() < 0.5:  # heads
            score += 2
        else:  # tails
            score *= 2
    return score

num_games = 10000
scores = [play_game() for _ in range(num_games)]
mean_score = sum(scores) / num_games
var_score = sum((score - mean_score)**2 for score in scores) / (num_games - 1)
stddev_score = math.sqrt(var_score)
stddev_mean = stddev_score / math.sqrt(num_games)

print(f"Expected standard deviation of the mean score: {stddev_mean:.6f}")

Expected standard deviation of the mean score: 0.229734


In [3]:
import random

def play_game():
    score = 0
    for i in range(7):
        if random.random() < 0.5:  # heads
            score += 2
        else:  # tails
            score *= 2
    return score

num_trials = 1000000
num_scores_over_40 = sum(play_game() > 40 for _ in range(num_trials))
prob_score_over_40 = num_scores_over_40 / num_trials

print(f"Probability of a score over 40: {prob_score_over_40:.6f}")

Probability of a score over 40: 0.234334


In [4]:
import random

def play_game():
    score = 0
    for i in range(20):
        if random.random() < 0.7:  # heads
            score += 2
        else:  # tails
            score *= 2
    return score

num_trials = 1000000
total_score = sum(play_game() for _ in range(num_trials))
expected_score = total_score / num_trials

print(f"Expected score after 20 flips: {expected_score:.6f}")

Expected score after 20 flips: 887.204726


In [5]:
import random
import math

def play_game():
    score = 0
    for i in range(20):
        if random.random() < 0.7:  # heads
            score += 2
        else:  # tails
            score *= 2
    return score

num_trials = 1000000
scores = [play_game() for _ in range(num_trials)]
mean_score = sum(scores) / num_trials
var_score = sum((score - mean_score)**2 for score in scores) / (num_trials - 1)
stddev_score = math.sqrt(var_score)

print(f"Standard deviation of the score after 20 flips: {stddev_score:.6f}")

Standard deviation of the score after 20 flips: 1757.269721


In [6]:
import numpy as np 
import pandas as pd
df = pd.read_csv('boston_311_calls.csv')
df.shape
# Calculate the number of calls that deal with traffic
traffic_calls = df[df['case_title'].str.contains('traffic', case=False, na=False)]
num_traffic_calls = len(traffic_calls)

# Calculate the total number of calls
num_total_calls = len(df)

# Calculate the fraction of calls that deal with traffic
fraction_traffic_calls = num_traffic_calls / num_total_calls

print(f"Fraction of calls dealing with traffic: {fraction_traffic_calls:.6f}")
# Clean up the police_district column
df['police_district'] = df['police_district'].str.replace('-', '')

# Convert the open date column to datetime format
df['open_dt'] = pd.to_datetime(df['open_dt'])

# Group the data by police district and year, and count the number of calls
calls_per_district_year = df.groupby([df['police_district'], df['open_dt'].dt.year])['case_enquiry_id'].count()

# Calculate the average annual number of calls per police district
avg_calls_per_district_year = calls_per_district_year.groupby('police_district').mean()

median_avg_calls_per_district_year = avg_calls_per_district_year.median()


print(f"Median of the average annual number of calls per police district: {median_avg_calls_per_district_year:.6f}")

zip = pd.read_csv('boston_population_by_zip.csv')
from scipy.stats import pearsonr

# Extract the ZIP code from the "location" column of the Boston dataset
df['zip_code'] = df['location'].str.extract(r'(\d{5})', expand=False)

# Count the number of calls per ZIP code
calls_per_zip = df.groupby('zip_code')['case_enquiry_id'].count()

# Merge the calls_per_zip and population dataframes on the zip_code column
#df_merged = pd.merge(calls_per_zip, zip, how='left', left_index=True, right_on='zip_code')

zip['zip_code'] = zip['zip_code'].astype(str)
df['zip_code'] = df['zip_code'].astype(str)

df_merged = pd.merge(df, zip, how='inner', left_on='zip_code', right_on='zip_code')


# Filter the merged DataFrame to include only ZIP codes with more than 100 calls
df_merged_filtered = df_merged[df_merged['case_enquiry_id'] < 100]

# Calculate the Pearson correlation coefficient
corr_coef, p_value = pearsonr(df_merged_filtered['case_enquiry_id'], df_merged_filtered['Population'])


print(f"Pearson correlation coefficient: {corr_coef:.6f}")
from scipy.stats import linregress

# Group the data by year and count the number of calls
calls_per_year = df.groupby(df['open_dt'].dt.year)['case_enquiry_id'].count()

# Fit a line of best fit to the data
slope, intercept, r_value, p_value, std_err = linregress(calls_per_year.index, calls_per_year.values)

# The slope represents the estimated increase in calls per year
print(f"Estimated increase in calls per year: {slope:.2f}")

# Filter the service requests data by ZIP code and request type
zip_code = "02128"
request_type = "Requests for Street Cleaning"
filtered_data = df[(df["location"].str.contains(zip_code)) & (df["type"] == request_type)]

garbage_data = pd.read_csv('boston_02128_garbage_schedule.csv')

# Define a function to calculate Euclidean distance between two points
def euclidean_distance(x1, y1, x2, y2):
    return np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

# Loop through each service request and find the closest garbage collection point
days_after_garbage_collection = []
for index, row in filtered_data.iterrows():
    request_latitude = row["latitude"]
    request_longitude = row["longitude"]
    min_distance = np.inf
    closest_garbage_day = ""
    for _, garbage_row in garbage_data.iterrows():
        garbage_latitude = garbage_row["latitude"]
        garbage_longitude = garbage_row["longitude"]
        distance = euclidean_distance(request_latitude, request_longitude, garbage_latitude, garbage_longitude)
        if distance < min_distance:
            min_distance = distance
            closest_garbage_day = garbage_row["trashday"]
    # Calculate the number of days between the street cleaning request and the closest garbage collection
    days_after_garbage_collection.append((row["open_dt"] - pd.to_datetime(closest_garbage_day)).days)

# Calculate the average number of days after a garbage collection that street cleaning requests are made
avg_days_after_garbage_collection = np.mean(days_after_garbage_collection)

print(f"On average, street cleaning requests in ZIP code {zip_code} are made {avg_days_after_garbage_collection:.2f} days after a garbage collection.")



In [7]:
df = pd.read_csv('boston_311_calls.csv')

In [8]:
df.shape

(1932760, 19)

In [10]:
df.columns

Index(['case_enquiry_id', 'open_dt', 'closed_dt', 'ontime', 'case_status',
       'case_title', 'subject', 'reason', 'type', 'department', 'location',
       'city_council_district', 'police_district', 'neighborhood', 'ward',
       'precinct', 'latitude', 'longitude', 'source'],
      dtype='object')

In [11]:
# Calculate the number of calls that deal with traffic
traffic_calls = df[df['case_title'].str.contains('traffic', case=False, na=False)]
num_traffic_calls = len(traffic_calls)

# Calculate the total number of calls
num_total_calls = len(df)

# Calculate the fraction of calls that deal with traffic
fraction_traffic_calls = num_traffic_calls / num_total_calls

print(f"Fraction of calls dealing with traffic: {fraction_traffic_calls:.6f}")

Fraction of calls dealing with traffic: 0.024412


In [14]:
from collections import Counter
from statistics import median
# Clean up the police_district column
df['police_district'] = df['police_district'].str.replace('-', '')

# Convert the open date column to datetime format
df['open_dt'] = pd.to_datetime(df['open_dt'])

# Group the data by police district and year, and count the number of calls
calls_per_district_year = df.groupby([df['police_district'], df['open_dt'].dt.year])['case_enquiry_id'].count()

# Calculate the average annual number of calls per police district
avg_calls_per_district_year = calls_per_district_year.groupby('police_district').mean()

median_avg_calls_per_district_year = avg_calls_per_district_year.median()


print(f"Median of the average annual number of calls per police district: {median_avg_calls_per_district_year:.6f}")

In [30]:
# Clean up the police_district column
df['police_district'] = df['police_district'].str.replace('-', '')

# Convert the open date column to datetime format
df['open_dt'] = pd.to_datetime(df['open_dt'])

# Group the data by police district and year, and count the number of calls
calls_per_district_year = df.groupby([df['police_district'], df['open_dt'].dt.year])['case_enquiry_id'].count()

# Calculate the average annual number of calls per police district
avg_calls_per_district_year = calls_per_district_year.groupby('police_district').mean()

median_avg_calls_per_district_year = avg_calls_per_district_year.median()


print(f"Median of the average annual number of calls per police district: {median_avg_calls_per_district_year:.6f}")

Median of the average annual number of calls per police district: 12400.583333


In [32]:
avg_calls_per_district_year.value_counts()

18125.000000    1
5379.416667     1
9867.583333     1
16004.583333    1
10619.500000    1
17950.250000    1
15095.500000    1
12767.916667    1
23007.750000    1
10677.250000    1
9466.000000     1
12033.250000    1
Name: case_enquiry_id, dtype: int64

In [40]:
zip = pd.read_csv('boston_population_by_zip.csv')
from scipy.stats import pearsonr

# Extract the ZIP code from the "location" column of the Boston dataset
df['zip_code'] = df['location'].str.extract(r'(\d{5})', expand=False)

# Count the number of calls per ZIP code
calls_per_zip = df.groupby('zip_code')['case_enquiry_id'].count()

# Merge the calls_per_zip and population dataframes on the zip_code column
#df_merged = pd.merge(calls_per_zip, zip, how='left', left_index=True, right_on='zip_code')

zip['zip_code'] = zip['zip_code'].astype(str)
df['zip_code'] = df['zip_code'].astype(str)

df_merged = pd.merge(df, zip, how='inner', left_on='zip_code', right_on='zip_code')


# Filter the merged DataFrame to include only ZIP codes with more than 100 calls
df_merged_filtered = df_merged[df_merged['case_enquiry_id'] < 100]

# Calculate the Pearson correlation coefficient
corr_coef, p_value = pearsonr(df_merged_filtered['case_enquiry_id'], df_merged_filtered['Population'])


print(f"Pearson correlation coefficient: {corr_coef:.6f}")
from scipy.stats import linregress

# Group the data by year and count the number of calls
calls_per_year = df.groupby(df['open_dt'].dt.year)['case_enquiry_id'].count()

# Fit a line of best fit to the data
slope, intercept, r_value, p_value, std_err = linregress(calls_per_year.index, calls_per_year.values)

# The slope represents the estimated increase in calls per year
print(f"Estimated increase in calls per year: {slope:.2f}")

# Filter the service requests data by ZIP code and request type
zip_code = "02128"
request_type = "Requests for Street Cleaning"
filtered_data = df[(df["location"].str.contains(zip_code)) & (df["type"] == request_type)]

# Define a function to calculate Euclidean distance between two points
def euclidean_distance(x1, y1, x2, y2):
    return np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

# Loop through each service request and find the closest garbage collection point
days_after_garbage_collection = []
for index, row in filtered_data.iterrows():
    request_latitude = row["latitude"]
    request_longitude = row["longitude"]
    min_distance = np.inf
    closest_garbage_day = ""
    for _, garbage_row in garbage_data.iterrows():
        garbage_latitude = garbage_row["latitude"]
        garbage_longitude = garbage_row["longitude"]
        distance = euclidean_distance(request_latitude, request_longitude, garbage_latitude, garbage_longitude)
        if distance < min_distance:
            min_distance = distance
            closest_garbage_day = garbage_row["trashday"]
    # Calculate the number of days between the street cleaning request and the closest garbage collection
    days_after_garbage_collection.append((row["open_dt"] - pd.to_datetime(closest_garbage_day)).days)

# Calculate the average number of days after a garbage collection that street cleaning requests are made
avg_days_after_garbage_collection = np.mean(days_after_garbage_collection)

print(f"On average, street cleaning requests in ZIP code {zip_code} are made {avg_days_after_garbage_collection:.2f} days after a garbage collection.")


In [41]:
zip.shape

(534, 2)

In [42]:
zip.columns

Index(['zip_code', 'Population'], dtype='object')

In [43]:
df.head()

,case_enquiry_id,open_dt,closed_dt,ontime,case_status,case_title,subject,reason,type,department,location,city_council_district,police_district,neighborhood,ward,precinct,latitude,longitude,source,zip_code
0,101001873350,2016-08-02 07:25:00,2016-11-01 11:43:30,OVERDUE,Closed,Graffiti Removal,Property Management,Graffiti,Graffiti Removal,PROP,218-230 Congress St Boston MA 02110,1.0,A1,Downtown / Financial District,Ward 3,0306,42.3545,-71.0542,Citizens Connect App,02110
1,101001168147,2014-09-09 13:57:10,2014-09-15 16:38:34,ONTIME,Closed,Schedule Bulk Item Pickup,Public Works Department,Sanitation,Schedule a Bulk Item Pickup SS,PWDx,106 Greenbrier St Dorchester MA 02124,4.0,C11,Dorchester,Ward 17,1703,42.2965,-71.0702,Self Service,02124
2,101000874384,2013-07-07 17:43:55,2013-07-09 14:07:53,ONTIME,Closed,Street Light Outages,Public Works Department,Street Lights,Street Light Outages,PWDx,INTERSECTION of Mountain Ave & Woodrow Ave Do...,4.0,B3,Greater Mattapan,Ward 14,1410,42.2868,-71.0858,Constituent Call,NaN
3,101001225622,2014-12-01 09:51:44,2014-12-02 16:37:16,ONTIME,Closed,Schedule a Bulk Item Pickup,Public Works Department,Sanitation,Schedule a Bulk Item Pickup,PWDx,106 Maplewood St West Roxbury MA 02132,6.0,E5,West Roxbury,Ward 20,2005,42.2731,-71.1507,Constituent Call,02132
4,101003332650,2020-06-29 23:01:00,NaN,OVERDUE,Open,Unsatisfactory Living Conditions,Inspectional Services,Housing,Unsatisfactory Living Conditions,ISD,63 Woodstock Ave Brighton MA 02135,9.0,D14,Allston / Brighton,Ward 21,2108,42.3463,-71.1368,Constituent Call,02135


In [52]:
from scipy.stats import pearsonr

# Extract the ZIP code from the "location" column of the Boston dataset
df['zip_code'] = df['location'].str.extract(r'(\d{5})', expand=False)

# Count the number of calls per ZIP code
calls_per_zip = df.groupby('zip_code')['case_enquiry_id'].count()

# Merge the calls_per_zip and population dataframes on the zip_code column
#df_merged = pd.merge(calls_per_zip, zip, how='left', left_index=True, right_on='zip_code')

zip['zip_code'] = zip['zip_code'].astype(str)
df['zip_code'] = df['zip_code'].astype(str)

df_merged = pd.merge(df, zip, how='inner', left_on='zip_code', right_on='zip_code')


# Filter the merged DataFrame to include only ZIP codes with more than 100 calls
df_merged_filtered = df_merged[df_merged['case_enquiry_id'] < 100]

# Calculate the Pearson correlation coefficient
corr_coef, p_value = pearsonr(df_merged_filtered['case_enquiry_id'], df_merged_filtered['Population'])


print(f"Pearson correlation coefficient: {corr_coef:.6f}")

ValueError: x and y must have length at least 2.

In [51]:
df_merged_filtered

,case_enquiry_id,open_dt,closed_dt,ontime,case_status,case_title,subject,reason,type,department,...,city_council_district,police_district,neighborhood,ward,precinct,latitude,longitude,source,zip_code,Population


In [53]:
from scipy.stats import linregress

# Group the data by year and count the number of calls
calls_per_year = df.groupby(df['open_dt'].dt.year)['case_enquiry_id'].count()

# Fit a line of best fit to the data
slope, intercept, r_value, p_value, std_err = linregress(calls_per_year.index, calls_per_year.values)

# The slope represents the estimated increase in calls per year
print(f"Estimated increase in calls per year: {slope:.2f}")

Estimated increase in calls per year: 15012.82


In [60]:
# Filter the service requests data by ZIP code and request type
zip_code = "02128"
request_type = "Requests for Street Cleaning"
filtered_data = df[(df["location"].str.contains(zip_code)) & (df["type"] == request_type)]

In [65]:
garbage_data = pd.read_csv('boston_02128_garbage_schedule.csv')

In [66]:
garbage_data.head()

,sam_address_id,zip_code,latitude,longitude,trashday
0,1117,2128,42.385440,-71.017280,Thursday
1,1118,2128,42.385440,-71.017280,Thursday
2,1119,2128,42.385440,-71.017280,Thursday
3,1120,2128,42.385410,-71.017080,Thursday
4,1122,2128,42.385342,-71.016721,Thursday


In [75]:
df['open_dt'].dtype

dtype('<M8[ns]')

In [67]:
# Define a function to calculate Euclidean distance between two points
def euclidean_distance(x1, y1, x2, y2):
    return np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

# Loop through each service request and find the closest garbage collection point
days_after_garbage_collection = []
for index, row in filtered_data.iterrows():
    request_latitude = row["latitude"]
    request_longitude = row["longitude"]
    min_distance = np.inf
    closest_garbage_day = ""
    for _, garbage_row in garbage_data.iterrows():
        garbage_latitude = garbage_row["latitude"]
        garbage_longitude = garbage_row["longitude"]
        distance = euclidean_distance(request_latitude, request_longitude, garbage_latitude, garbage_longitude)
        if distance < min_distance:
            min_distance = distance
            closest_garbage_day = garbage_row["trashday"]
    # Calculate the number of days between the street cleaning request and the closest garbage collection
    days_after_garbage_collection.append((row["open_dt"] - pd.to_datetime(closest_garbage_day)).days)

# Calculate the average number of days after a garbage collection that street cleaning requests are made
avg_days_after_garbage_collection = np.mean(days_after_garbage_collection)

print(f"On average, street cleaning requests in ZIP code {zip_code} are made {avg_days_after_garbage_collection:.2f} days after a garbage collection.")


In [76]:
# Loop through each service request and find the closest garbage collection point
days_after_garbage_collection = []
for index, row in filtered_data.iterrows():
    request_latitude = row["latitude"]
    request_longitude = row["longitude"]
    min_distance = np.inf
    closest_garbage_day = ""
    for _, garbage_row in garbage_data.iterrows():
        garbage_latitude = garbage_row["latitude"]
        garbage_longitude = garbage_row["longitude"]
        distance = euclidean_distance(request_latitude, request_longitude, garbage_latitude, garbage_longitude)
        if distance < min_distance:
            min_distance = distance
            closest_garbage_day = garbage_row["trashday"]
    # Calculate the number of days between the street cleaning request and the closest garbage collection
    days_after_garbage_collection.append((row["open_dt"] - pd.to_datetime(closest_garbage_day)).days)

# Calculate the average number of days after a garbage collection that street cleaning requests are made
avg_days_after_garbage_collection = np.mean(days_after_garbage_collection)

print(f"On average, street cleaning requests in ZIP code {zip_code} are made {avg_days_after_garbage_collection:.2f} days after a garbage collection.")


OutOfBoundsDatetime: Out of bounds nanosecond timestamp: 1-01-01 00:00:00